# 52e. Figure 4 with the TF-IDF reference, Figure 3 with the legend off the bars, and a faster commit step

Notebook 52d re-exported Figures 3, 4 and 6 at 600 dpi but left two defects. Figure 4 lost the TF-IDF
reference curve (the manuscript caption lists it and Table 4 reports its AUROC of 0.671). Figure 3 placed
the legend of the bar panel over the `ignore` and `combined` bars.

This notebook redraws both from the same caches. The TF-IDF reference is not cached as scores: notebook 18
fits it in seconds from `data/structural/train.jsonl` (TF-IDF, unigrams and bigrams, logistic regression),
so Stage B refits it the same way and checks its AUROC against `reports/structural_detector_eval.csv`
before drawing. No neural model is loaded and nothing is re-scored.

Stage A also rewrites `tools/commit_cell.py` so that output stripping touches only the notebooks that
`git status` reports as changed, in-process, instead of running `nbconvert` once per notebook on all of
them (that is what made the last commit take sixteen minutes). Everything else in the commit step is
unchanged.

**Outputs.** `tools/commit_cell.py`, `figures/opi_second_source.png` (Figure 3),
`figures/structural_detector_roc.png` (Figure 4), both at 600 dpi.

In [ ]:
# === SESSION BOOTSTRAP (run first): no GPU needed ===
from google.colab import drive
drive.mount('/content/drive')
import os, sys
DRIVE_ROOT = '/content/drive/MyDrive/PICALIB_Research'
REPO_DIR   = os.path.join(DRIVE_ROOT, 'picalib-research')
!git config --global user.name  "Md Anas Biswas"
!git config --global user.email "anasbiswas@gmail.com"
!git config --global credential.helper "store --file={DRIVE_ROOT}/.git-credentials"
%cd {REPO_DIR}
sys.path.insert(0, 'src')
!git pull
import numpy as np, pandas as pd, json, importlib
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, roc_auc_score
DPI = 600
PAPER = {'structural (full)': 'structural (full)',
         'structural (ablation_no_twins)': 'structural (ablation, no twins)',
         'structural (full_sizematched)': 'structural (size-matched)'}
def nm(s): return PAPER.get(s, s)
# one colour per detector, shared by Figures 3 and 4 (matplotlib's default cycle, in this order)
ORDER = ['ProtectAI-v2', 'Prompt-Guard-2 (86M)', 'Prompt-Guard-2 (22M)', 'structural (full)',
         'structural (ablation, no twins)', 'structural (size-matched)', 'TF-IDF reference']
COLOR = {n: f'C{i}' for i, n in enumerate(ORDER)}
print('Session ready.')

## Stage A. Faster commit step: strip outputs of changed notebooks only

In [ ]:
%%writefile tools/commit_cell.py
"""Single commit step for this repo. Run from a notebook's final cell only:

    subprocess.run(["python", "tools/commit_cell.py", "message"])

Restores git identity and credentials from the Drive project folder, strips the outputs of every notebook
that git reports as changed, refuses to stage data, credential or oversized files, refuses any staged file
containing a secret token pattern, commits if anything is staged, and always pushes so unpushed commits
never remain local.
"""
from __future__ import annotations

import os
import re
import shutil
import subprocess
import sys
from pathlib import Path

DRIVE_PROJECT_ROOT = Path("/content/drive/MyDrive/PICALIB_Research")
REPO_ROOT = Path(os.environ.get("PICALIB_REPO_ROOT", str(DRIVE_PROJECT_ROOT / "picalib-research")))
GIT_NAME = "Md Anas Biswas"
GIT_EMAIL = "anasbiswas@gmail.com"
FORBIDDEN_NAMES = {".git-credentials", ".gitconfig", "kaggle.json", ".env"}
FORBIDDEN_PREFIXES = ("data/", "_bipia/", ".secrets/")
MAX_FILE_MB = 50
SECRET_PATTERNS = [
    re.compile(r"hf_[A-Za-z0-9]{20,}"),            # Hugging Face
    re.compile(r"ghp_[A-Za-z0-9]{30,}"),           # GitHub classic PAT
    re.compile(r"github_pat_[A-Za-z0-9_]{40,}"),   # GitHub fine-grained PAT
    re.compile(r"sk-[A-Za-z0-9]{32,}"),             # OpenAI-style keys
    re.compile(r"AKIA[0-9A-Z]{16}"),               # AWS access key id
    re.compile(r"AIza[0-9A-Za-z_\-]{35}"),         # Google API key
]
TEXT_EXT = {".py", ".ipynb", ".md", ".txt", ".json", ".csv", ".yaml", ".yml", ".toml", ".cfg", ".ini", ".sh"}


def run(args, check=True):
    r = subprocess.run(args, capture_output=True, text=True)
    if check and r.returncode != 0:
        raise SystemExit(f"{' '.join(args)}\n{r.stdout}{r.stderr}")
    return r.stdout


def changed_notebooks():
    """Notebooks that are modified, added or untracked. Committed notebooks are already stripped."""
    paths = []
    for line in run(["git", "status", "--porcelain", "--untracked-files=all", "--", "notebooks"]).splitlines():
        p = line[3:].strip().strip('"')
        if " -> " in p:
            p = p.split(" -> ")[-1]
        if p.endswith(".ipynb") and (REPO_ROOT / p).is_file():
            paths.append(REPO_ROOT / p)
    return sorted(set(paths))


def strip_outputs():
    import nbformat
    notebooks = changed_notebooks()
    for notebook in notebooks:
        nb = nbformat.read(notebook, as_version=4)
        for cell in nb.cells:
            if cell.cell_type == "code":
                cell["outputs"] = []
                cell["execution_count"] = None
        nb.metadata.pop("widgets", None)
        nbformat.write(nb, notebook)
    print(f"outputs: stripped {len(notebooks)} changed notebook(s)")


def secret_scan(paths):
    hits = []
    for p in paths:
        path = REPO_ROOT / p
        if path.suffix.lower() not in TEXT_EXT or not path.is_file():
            continue
        try:
            txt = path.read_text(encoding="utf-8", errors="ignore")
        except Exception:
            continue
        for pat in SECRET_PATTERNS:
            if pat.search(txt):
                hits.append(f"{p} ({pat.pattern[:14]}...)")
                break
    return hits


def main() -> None:
    if len(sys.argv) < 2 or not sys.argv[1].strip():
        raise SystemExit('usage: python tools/commit_cell.py "commit message"')
    message = sys.argv[1].strip()

    for dotfile in (".gitconfig", ".git-credentials"):
        source = DRIVE_PROJECT_ROOT / dotfile
        if source.exists():
            shutil.copy(source, Path.home() / dotfile)
    credentials = Path.home() / ".git-credentials"
    if credentials.exists():
        os.chmod(credentials, 0o600)

    os.chdir(REPO_ROOT)
    run(["git", "config", "user.name", GIT_NAME])
    run(["git", "config", "user.email", GIT_EMAIL])
    run(["git", "config", "credential.helper", "store"])

    strip_outputs()

    run(["git", "add", "-A"])
    staged = [l for l in run(["git", "diff", "--cached", "--name-only"]).splitlines() if l.strip()]
    bad = [p for p in staged if Path(p).name in FORBIDDEN_NAMES or p.startswith(FORBIDDEN_PREFIXES)]
    big = [p for p in staged if Path(p).exists() and Path(p).stat().st_size > MAX_FILE_MB * 1024 * 1024]
    secrets = secret_scan(staged)
    if bad or big or secrets:
        run(["git", "reset", "-q"], check=False)
        raise SystemExit(f"refused: forbidden={bad} oversized={big} secrets_in={secrets}")

    if staged:
        commit = subprocess.run(["git", "commit", "-m", message], capture_output=True, text=True)
        print(commit.stdout or commit.stderr)
    else:
        print("nothing new to commit; pushing any unpushed commits")

    push = subprocess.run(["git", "push", "origin", "main"], capture_output=True, text=True)
    print(push.stdout or push.stderr)
    print(run(["git", "log", "--oneline", "-1"]).strip())
    print(run(["git", "status", "--short"]).strip() or "working tree clean")


if __name__ == "__main__":
    main()


In [ ]:
import ast
src = open('tools/commit_cell.py').read(); ast.parse(src)
assert 'changed_notebooks' in src and 'nbconvert' not in src
print('tools/commit_cell.py rewritten: strips changed notebooks only')

## Stage B. Figure 4: all seven curves, including the TF-IDF reference (notebook 18's panel)

In [ ]:
import data_loaders, structural_train; importlib.reload(data_loaders); importlib.reload(structural_train)
from data_loaders import load_bipia_categorized, BIPIA_MALICIOUS, BIPIA_HIJACK
from structural_train import load_jsonl
bip = load_bipia_categorized(per_category=30, categories=BIPIA_MALICIOUS)
HIJ = bip.meta.isin(BIPIA_HIJACK).values; HOST = bip.label.values == 0
print('BIPIA rows:', len(bip), '| hijack:', int(HIJ.sum()), '| hosts:', int(HOST.sum()))

TAGS18 = {'protectai_v2': 'ProtectAI-v2', 'prompt_guard_2': 'Prompt-Guard-2 (86M)', 'prompt_guard_2_22m': 'Prompt-Guard-2 (22M)'}
S18 = {}; missing = []
for t, lab in TAGS18.items():
    fp = f'data/score_{t}_bipia_cat.npy'
    if os.path.exists(fp): S18[lab] = np.load(fp)
    else: missing.append(fp)
for cfg in ['full', 'ablation_no_twins', 'full_sizematched']:
    fp = f'data/structural/scores/{cfg}_bipia.npy'
    if os.path.exists(fp): S18[nm(f'structural ({cfg})')] = np.load(fp)
    else: missing.append(fp)
assert not missing, 'missing cached scores:\n  ' + '\n  '.join(missing)

# the TF-IDF reference exactly as notebook 18 fits it (deterministic; no cached scores exist for it)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
train = load_jsonl('data/structural/train.jsonl')
vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=200000, sublinear_tf=True)
Xtr = vec.fit_transform([r['text'] for r in train])
lr = LogisticRegression(C=1.0, max_iter=2000).fit(Xtr, [r['label'] for r in train])
S18['TF-IDF reference'] = lr.predict_proba(vec.transform(bip.text.tolist()))[:, 1]

for k, v in S18.items(): assert len(v) == len(bip), f'{k}: {len(v)} vs {len(bip)}'
y = np.r_[np.ones(int(HIJ.sum())), np.zeros(int(HOST.sum()))]
EV = pd.read_csv('reports/structural_detector_eval.csv').set_index('model')
auc_now = roc_auc_score(y, np.r_[S18['TF-IDF reference'][HIJ], S18['TF-IDF reference'][HOST]])
auc_rep = float(EV.loc['TF-IDF reference', 'AUROC_hijack'])
print(f'TF-IDF reference AUROC now {auc_now:.4f}, reported {auc_rep:.4f}')
assert abs(auc_now - auc_rep) < 0.005, 'the refitted TF-IDF reference does not reproduce Table 4; stop and report'
print('scorers:', list(S18))

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 5.5))
for name in ORDER:
    p = S18[name]; pp = np.r_[p[HIJ], p[HOST]]; f, t, _ = roc_curve(y, pp)
    ax.plot(f, t, lw=2 if name == 'structural (full)' else 1.2, color=COLOR[name], label=f'{name} (AUROC {roc_auc_score(y, pp):.2f})')
ax.plot([0, 1], [0, 1], 'k--', lw=.8); ax.axvline(0.01, color='grey', ls=':', lw=.8)
ax.set_xlabel('false-positive rate on BIPIA host documents'); ax.set_ylabel('true-positive rate on hijack injections')
ax.set_title('Hijack injection versus benign host documents'); ax.legend(fontsize=7, loc='lower right'); ax.grid(alpha=.3)
plt.tight_layout(); plt.savefig('figures/structural_detector_roc.png', dpi=DPI, bbox_inches='tight'); plt.show()
print('saved figures/structural_detector_roc.png at', DPI, 'dpi with', len(ORDER), 'curves')

## Stage C. Figure 3: legend outside the bar panel, one colour per detector in both panels (notebook 19's panel)

In [ ]:
OPI = json.load(open('data/opi/opi_eval.json'))
R = OPI['records']; TXT = [r['text'] for r in R]; Y = np.array([r['label'] for r in R])
S19 = {}; missing = []
for t, lab in TAGS18.items():
    fp = f'data/opi/score_{t}.npy'
    if os.path.exists(fp): S19[lab] = np.load(fp)
    else: missing.append(fp)
for cfg in ['full', 'ablation_no_twins', 'full_sizematched']:
    fp = f'data/opi/score_structural_{cfg}.npy'
    if os.path.exists(fp): S19[nm(f'structural ({cfg})')] = np.load(fp)
    else: missing.append(fp)
assert not missing, 'missing cached scores:\n  ' + '\n  '.join(missing)
for k, v in S19.items(): assert len(v) == len(TXT), f'{k}: {len(v)} vs {len(TXT)}'

BS = pd.read_csv('reports/opi_by_strategy.csv')
STRATEGIES = ['naive', 'escape', 'fake_completion', 'ignore', 'combined']
piv = BS.pivot_table(index='strategy', columns='model', values='FNR_guar').reindex(STRATEGIES)
piv.columns = [nm(c) for c in piv.columns]; piv = piv[ORDER]
piv.index = [s.replace('_', ' ') for s in piv.index]

fig, axes = plt.subplots(1, 2, figsize=(13, 5), gridspec_kw={'width_ratios': [1.05, 1]})
for name in [n for n in ORDER if n in S19]:
    f, t, _ = roc_curve(Y, S19[name])
    axes[0].plot(f, t, lw=2 if name == 'structural (full)' else 1.2, color=COLOR[name], label=f'{name} ({roc_auc_score(Y, S19[name]):.2f})')
axes[0].plot([0, 1], [0, 1], 'k--', lw=.8); axes[0].axvline(0.01, color='grey', ls=':', lw=.8)
axes[0].set_xlabel('false-positive rate on matched benign data'); axes[0].set_ylabel('true-positive rate on injected data')
axes[0].set_title('Second source (Open-Prompt-Injection construction)'); axes[0].legend(fontsize=7, loc='lower right'); axes[0].grid(alpha=.3)
piv.plot.bar(ax=axes[1], width=.8, color=[COLOR[c] for c in piv.columns])
axes[1].set_ylabel('FNR at guaranteed 1 percent FPR'); axes[1].set_xlabel('strategy'); axes[1].set_title('By attack strategy')
axes[1].legend(fontsize=7, loc='upper left', bbox_to_anchor=(1.01, 1.0)); axes[1].grid(axis='y', alpha=.3); axes[1].set_ylim(0, 1.05)
plt.tight_layout(); plt.savefig('figures/opi_second_source.png', dpi=DPI, bbox_inches='tight'); plt.show()
print('saved figures/opi_second_source.png at', DPI, 'dpi')

## Stage D. Size check

In [ ]:
from PIL import Image
for fp in ['figures/opi_second_source.png', 'figures/structural_detector_roc.png']:
    im = Image.open(fp); print(f'{fp}: {im.size[0]} x {im.size[1]} px, dpi {im.info.get("dpi")}')
    assert im.size[0] >= 3000, f'{fp} is still small: re-run the stage that writes it'
print('both figures at 600 dpi')

In [ ]:
# === STANDARD COMMIT CELL ===
import os, subprocess
os.chdir(REPO_DIR)
branch = subprocess.run(["git", "rev-parse", "--abbrev-ref", "HEAD"], capture_output=True, text=True).stdout.strip()
assert branch == "main", f"the repository is on branch {branch!r}, not main: switch to main before committing"
r = subprocess.run(["python", "tools/commit_cell.py",
                    "nb52e: Figure 4 with the TF-IDF reference restored, Figure 3 with the legend off the bars; commit step strips changed notebooks only"],
                   capture_output=True, text=True)
print(r.stdout); print(r.stderr)